In [7]:
import cv2
import os
import numpy as np
import json

# Load Haar Cascade face detector
face_detector = cv2.CascadeClassifier(
    cv2.data.haarcascades +
    "haarcascade_frontalface_default.xml"
)

# Create LBPH recognizer
recognizer = cv2.face.LBPHFaceRecognizer_create()

# Folder containing known faces
dataset_path = "known_faces"

faces = []
labels = []
label_names = {}

current_label = 0

# Read each person's folder
for person_name in os.listdir(dataset_path):

    person_path = os.path.join(
        dataset_path,
        person_name
    )

    if not os.path.isdir(person_path):
        continue

    print("Processing:", person_name)

    # Store label → person name
    label_names[current_label] = person_name

    # Read all images of this person
    for image_name in os.listdir(person_path):

        image_path = os.path.join(
            person_path,
            image_name
        )

        image = cv2.imread(image_path)

        if image is None:
            continue

        # Convert to grayscale
        gray = cv2.cvtColor(
            image,
            cv2.COLOR_BGR2GRAY
        )

        # Detect face
        detected_faces = face_detector.detectMultiScale(
            gray,
            scaleFactor=1.05,
            minNeighbors=5,
            minSize=(50, 50)
        )

        # Add detected faces to training data
        for (x, y, w, h) in detected_faces:

            face = gray[y:y+h, x:x+w]

            faces.append(face)
            labels.append(current_label)

    current_label += 1


# Check if faces were found
if len(faces) == 0:

    print("No faces found in the database!")

else:

    # Train model
    recognizer.train(
        faces,
        np.array(labels)
    )

    # Save model
    recognizer.save("face_model.yml")

    # Save names
    with open("labels.json", "w") as file:
        json.dump(label_names, file)

    print("\nTraining completed!")
    print("Model saved as face_model.yml")
    print("Labels saved as labels.json")
    print("Training faces:", len(faces))

Processing: Issac
Processing: Yohan

Training completed!
Model saved as face_model.yml
Labels saved as labels.json
Training faces: 181


In [16]:
import cv2
import json

# Load face detector
face_detector = cv2.CascadeClassifier(
    cv2.data.haarcascades +
    "haarcascade_frontalface_default.xml"
)

# Load trained model
recognizer = cv2.face.LBPHFaceRecognizer_create()
recognizer.read("face_model.yml")

# Load person names
with open("labels.json", "r") as file:
    label_names = json.load(file)

# Convert JSON keys back to integers
label_names = {
    int(key): value
    for key, value in label_names.items()
}


# Function to resize image for display
def resize_for_display(image, max_width=800, max_height=550):

    height, width = image.shape[:2]

    scale = min(
        max_width / width,
        max_height / height,
        1
    )

    new_width = int(width * scale)
    new_height = int(height * scale)

    return cv2.resize(
        image,
        (new_width, new_height)
    )


# Function to recognize face
def recognize_face(face):

    label, confidence = recognizer.predict(face)

    if confidence < 65:
        return label_names[label]
    else:
        return "Unknown"


# Choose input
print("1. Image")
print("2. Webcam")

choice = input("Enter your choice: ")


# ================= IMAGE =================

if choice == "1":

    image_path = input("Enter image path: ")

    image = cv2.imread(image_path)

    if image is None:
        print("Image not found!")
    else:

        gray = cv2.cvtColor(
            image,
            cv2.COLOR_BGR2GRAY
        )

        faces = face_detector.detectMultiScale(
            gray,
            scaleFactor=1.05,
            minNeighbors=6,
            minSize=(50, 50)
        )

        for (x, y, w, h) in faces:

            face = gray[y:y+h, x:x+w]

            name = recognize_face(face)

            cv2.rectangle(
                image,
                (x, y - 45),
                (x + w, y),
                (0, 255, 0),
                -1
            )
            
            cv2.putText(
                image,
                name,
                (x + 5, y - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                2,
                (0, 0, 0),
                3
            )

        print("Faces detected:", len(faces))

        display_image = resize_for_display(image)

        cv2.imshow(
            "Face Recognition",
            display_image
        )

        cv2.waitKey(0)
        cv2.destroyAllWindows()


# ================= WEBCAM =================

elif choice == "2":

    cap = cv2.VideoCapture(0)

    if not cap.isOpened():
        print("Unable to open webcam!")
    else:

        while True:

            ret, frame = cap.read()

            if not ret:
                break

            gray = cv2.cvtColor(
                frame,
                cv2.COLOR_BGR2GRAY
            )

            faces = face_detector.detectMultiScale(
                gray,
                scaleFactor=1.05,
                minNeighbors=6,
                minSize=(50, 50)
            )

            for (x, y, w, h) in faces:

                face = gray[y:y+h, x:x+w]

                name = recognize_face(face)

                cv2.rectangle(
                    image,
                    (x, y - 45),
                    (x + w, y),
                    (0, 255, 0),
                    -1
                )
                
                cv2.putText(
                    image,
                    name,
                    (x + 5, y - 10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    2,
                    (0, 0, 0),
                    3
                )

            display_frame = resize_for_display(frame)

            cv2.imshow(
                "Face Recognition",
                display_frame
            )

            # ESC to exit
            if cv2.waitKey(1) == 27:
                break

        cap.release()
        cv2.destroyAllWindows()

else:
    print("Invalid choice!")

1. Image
2. Webcam


Enter your choice:  2
